# Carry Capacity and the Conditional Cut Law

### Four laws of the two-operand adder recovered as one corner of a three-parameter family

**Dean Kulik** — QuHarmonics Research Group — ORCID 0009-0003-3128-8828

---

A previous paper established a cut law for the two-operand binary adder: the number of
Myhill–Nerode states a reader must carry on a cut `S` factorizes as a product over the
maximal runs of `S`, with per-run factors 1, 2, L+1 and 2L+1 according to which ends of
the word the run touches. This notebook reports that the law is not about runs. It is
about whether the unread region between two runs is large enough to sever their
dependence, and the two-operand binary case is the corner in which that condition is
vacuously satisfied and therefore invisible.

The machine is generalised in three coordinates — the radix `b`, the operand count `m`,
and whether the unresolved value sits on the tape or in the carry register. Every number
below is produced by an executed cell. The library cell is complete; this notebook
imports nothing from any earlier deliverable.

In [2]:
"""
Carry capacity and the conditional cut law -- complete verification library.

Self-contained: nothing here imports from any earlier deliverable.  Everything the paper
quotes is computed from this file.

The machine family has three coordinates:

    b   the radix
    m   the number of operands summed at each position
    where the value is loaded: on the tape, or into the carry register

and one loop:

    s_q = (t_q + c_q) mod b ,      c_{q+1} = floor((t_q + c_q) / b)

with t_q the digit sum at position q, ranging over 0 .. m(b-1).  The published work is
the corner b = 2, m = 2, value on the tape.
"""
import itertools, math
from collections import defaultdict


# ===================================================================== cuts and runs
def cuts(n):
    """every subset of positions, as sorted tuples"""
    return [tuple(S) for k in range(n + 1) for S in itertools.combinations(range(n), k)]


def runs_of(S, n):
    """maximal runs of READ positions, as (l, r) inclusive"""
    out = []
    for p in sorted(S):
        if out and out[-1][1] == p - 1:
            out[-1][1] = p
        else:
            out.append([p, p])
    return [tuple(r) for r in out]


def gaps_of(S, n):
    """lengths of the maximal runs of UNREAD positions"""
    U = [q for q in range(n) if q not in set(S)]
    return [len(list(g)) for _, g in itertools.groupby(enumerate(U), lambda t: t[1] - t[0])]


def contact(l, r, n):
    """which machine boundaries a run touches"""
    if l == 0 and r == n - 1: return "full"
    if l == 0:                return "bot"
    if r == n - 1:            return "top"
    return "int"


def boundary(S, n):
    """the cut's boundary positions, with virtual walls at -1 and n"""
    chi = [0] + [1 if p in set(S) else 0 for p in range(n)] + [0]
    return tuple(p for p in range(n + 1) if chi[p] != chi[p + 1])


def loc(S, n):
    """the largest block size the cut stays aligned to: gcd of its boundary and n"""
    g = n
    for p in boundary(S, n):
        g = math.gcd(g, p)
    return g


def divisors(n):
    return [k for k in range(1, n + 1) if n % k == 0]


def lift(T, k):
    """a coarse position q covers fine positions qk .. qk+k-1"""
    return tuple(sorted(p for q in T for p in range(q * k, q * k + k)))


# ===================================================================== the machines
def line_out(t, b, n):
    """the severed machine: c_0 = 0 below, carry out discarded above"""
    c = 0; s = []
    for q in range(n):
        v = t[q] + c; s.append(v % b); c = v // b
    return frozenset({tuple(s)})


def free_out(t, b, n, m=2):
    """the same machine with the origin removed: c_0 unknown"""
    outs = set()
    for c0 in range(m):
        c = c0; s = []
        for q in range(n):
            v = t[q] + c; s.append(v % b); c = v // b
        outs.add(tuple(s))
    return frozenset(outs)


def ring_out(t, b, n, m=2):
    """end-around carry: every carry assignment consistent on the circle"""
    outs = set()
    for c0 in range(m):
        c = c0; s = []
        for q in range(n):
            v = t[q] + c; s.append(v % b); c = v // b
        if c == c0:
            outs.add(tuple(s))
    return frozenset(outs)


def uncoupled_out(t, b, n, m=2):
    """no coupling between positions at all"""
    return frozenset({tuple(t)})


# ===================================================================== the reader
def mn_class(n, b, m, S, data, out=line_out):
    """Myhill-Nerode class of a reading: which output positions are still pending,
    and how they vary with the unread data."""
    S = sorted(S); U = [p for p in range(n) if p not in S]
    hi = m * (b - 1) + 1
    rows = []
    for uv in itertools.product(range(hi), repeat=len(U)):
        t = dict(data); t.update(dict(zip(U, uv)))
        rows.append(out([t[q] for q in range(n)], b, n))
    pend = tuple(i for i in range(n) if len({o[i] for r in rows for o in r}) > 1)
    return (pend, tuple(frozenset(tuple(o[i] for i in pend) for o in r) for r in rows))


def F(n, b, m, S, out=line_out):
    """number of distinct reader states on cut S"""
    S = sorted(S); hi = m * (b - 1) + 1
    return len({mn_class(n, b, m, S, dict(zip(S, kv)), out)
                for kv in itertools.product(range(hi), repeat=len(S))})


def F_product(n, b, m, S, out=line_out):
    """what the product law predicts: the product of the single-run values"""
    p = 1
    for (l, r) in runs_of(set(S), n):
        p *= F(n, b, m, list(range(l, r + 1)), out)
    return p


# ===================================================================== capacity
def transfer(ts, c, b):
    """the carry transfer a gap word induces"""
    for t in ts:
        c = (t + c) // b
    return c


def constants(b, m, g):
    """the constant carry transfers a gap of g positions can realise"""
    hi = m * (b - 1) + 1
    out = set()
    for ts in itertools.product(range(hi), repeat=g):
        vals = {transfer(ts, c, b) for c in range(m)}
        if len(vals) == 1:
            out.add(next(iter(vals)))
    return sorted(out)


def gstar(b, m):
    """the predicted capacity threshold"""
    return math.ceil(math.log(m, b) - 1e-12)


def reach(b, m, L):
    """how many carry values a bottom-anchored run of length L can BUILD from c_0 = 0"""
    r = {0}
    for _ in range(L):
        r = {(t + c) // b for c in r for t in range(m * (b - 1) + 1)}
    return len(r)


def tail_cap(b, m, tail):
    """how many carry values an unread tail of that length can DISTINGUISH"""
    return min(m, b ** tail)


def f_bot_law(b, m, n, L):
    """the two-sided bottom factor: build capacity capped by distinguishing capacity"""
    return min(reach(b, m, L), tail_cap(b, m, n - L))


def transfer_maps(b, m, g, extra_reset=False):
    """every carry-transfer map on {0..m-1} that a g-position gap can realise.
    extra_reset adds one admissible symbol whose transfer is c -> 0."""
    syms = list(range(m * (b - 1) + 1)) + (["R"] if extra_reset else [])
    out = set()
    for ts in itertools.product(syms, repeat=g):
        f = []
        for c in range(m):
            for t in ts:
                c = 0 if t == "R" else (t + c) // b
            f.append(c)
        out.add(tuple(f))
    return out


def zeros_transfer(b, m, g):
    """the all-zeros gap word's transfer: c -> floor(c / b^g)"""
    return tuple(c // b ** g for c in range(m))


def max_carry(b, m, iters=64):
    """the carry alphabet, by iterating the worst case to a fixed point"""
    c = 0
    for _ in range(iters):
        c = (m * (b - 1) + c) // b
    return c


# ===================================================================== motion
def digits(x, b, n):
    return [(x // b ** i) % b for i in range(n)]


def carry_walk(x, y, b, n):
    """the carry trajectory of x + y, LSB first, including c_0 and c_n"""
    c = 0; w = [0]
    for q in range(n):
        v = (x % b) + (y % b) + c
        c = v // b; w.append(c)
        x //= b; y //= b
    return tuple(w)


def loop(tape, c0, b, n):
    """the one loop, either loading.  returns the emitted digits and the residue."""
    c = c0; out = []
    for q in range(n):
        v = tape[q] + c
        out.append(v % b); c = v // b
    return out, c


def blocks(x, k, N):
    return tuple((x >> (i * k)) & ((1 << k) - 1) for i in range(N // k))


def period(x, N):
    """the smallest block size at which x is a repdigit"""
    for k in divisors(N):
        if len(set(blocks(x, k, N))) == 1:
            return k
    return N


def orbit_len(x, N):
    """orbit length of x under the reading-origin rotation, x -> 2x mod 2^N - 1"""
    M = (1 << N) - 1
    if M == 0: return 1
    o = 1; y = (2 * x) % M
    while y != x:
        y = (2 * y) % M; o += 1
    return o


# ===================================================================== filtration
class Filtration:
    """A''' lives here: a finite coordinate set, a quotient per subset."""

    def __init__(self, n, alpha, cls, name):
        self.n, self.alpha, self.cls, self.name = n, alpha, cls, name

    def assignments(self, S):
        S = tuple(sorted(S))
        for kv in itertools.product(range(self.alpha), repeat=len(S)):
            yield dict(zip(S, kv))

    def Q(self, S):
        return {self.cls(tuple(sorted(S)), a) for a in self.assignments(S)}

    def surjective(self):
        """H1: restriction Q_S -> Q_{S-p} is onto"""
        for sz in range(1, self.n + 1):
            for S in itertools.combinations(range(self.n), sz):
                for p in S:
                    T = tuple(q for q in S if q != p)
                    img = {self.cls(T, {q: a[q] for q in T}) for a in self.assignments(S)}
                    if img != self.Q(T):
                        return False
        return True

    def full_stateless(self):
        """H2: the full cut is a single class"""
        return len(self.Q(tuple(range(self.n)))) == 1

    def nontrivial(self):
        return max(len(self.Q(S)) for sz in range(self.n + 1)
                   for S in itertools.combinations(range(self.n), sz)) > 1

    def backward_functional(self):
        """is un-reading a function of the state?"""
        for sz in range(1, self.n + 1):
            for S in itertools.combinations(range(self.n), sz):
                by = defaultdict(set)
                for a in self.assignments(S):
                    by[self.cls(S, a)].add(tuple(sorted(a.items())))
                for p in S:
                    T = tuple(q for q in S if q != p)
                    for members in by.values():
                        img = {self.cls(T, {q: dict(a)[q] for q in T}) for a in members}
                        if len(img) > 1:
                            return False
        return True

    def signature(self):
        return (self.surjective(), self.full_stateless(),
                self.nontrivial(), self.backward_functional())


def adder_filtration(n, free):
    """the adder as a filtration, with and without its origin"""
    PAIRS = [(0, 0), (0, 1), (1, 0), (1, 1)]

    def cls(S, a):
        S = tuple(sorted(S)); U = [p for p in range(n) if p not in S]
        outs = []
        for uv in itertools.product(PAIRS, repeat=len(U)):
            x = y = 0
            for p in S:
                u, v = PAIRS[a[p]]; x |= u << p; y |= v << p
            for p, (u, v) in zip(U, uv):
                x |= u << p; y |= v << p
            for c0 in ((0, 1) if free else (0,)):
                s = (x + y + c0) % (1 << n)
                outs.append(tuple((s >> i) & 1 for i in range(n)))
        pend = tuple(i for i in range(n) if len({o[i] for o in outs}) > 1)
        return (pend, tuple(tuple(o[i] for i in pend) for o in outs))

    return Filtration(n, 4, cls, f"adder n={n}, {'free' if free else 'pinned'} c_0")


def coord0_filtration(n):
    """the H2 witness that works: Q_S is the value of coordinate 0 when 0 is read,
    and one class otherwise.  A genuine quotient of partial assignments; it simply is
    not the residual quotient of a closed machine."""
    return Filtration(n, 2, lambda S, a: a[0] if 0 in S else "*",
                      f"coord-0 quotient, n={n}")


def external_filtration(n):
    """the 'machine' version offered in v1.4 as a non-contrived witness.  It is not a
    witness: every |Q_S| is 1, so it is trivial and demonstrates nothing."""
    def cls(S, a):
        S = tuple(sorted(S)); U = [p for p in range(n) if p not in S]
        outs = []
        for uv in itertools.product(range(2), repeat=len(U)):
            v = dict(a); v.update(dict(zip(U, uv)))
            for ext in (0, 1):
                outs.append((v[0], ext))
        pend = tuple(i for i in range(2) if len({o[i] for o in outs}) > 1)
        return (pend, tuple(tuple(o[i] for i in pend) for o in outs))

    return Filtration(n, 2, cls, f"external-coordinate machine, n={n}")

## 1. What is claimed and what is withdrawn

No computation in the previous work is withdrawn. Every number there is correct in its
stated domain. What is withdrawn is the scope those numbers were given.

| stated as | what it is |
|---|---|
| `F(S) = ∏ f(R)` for all `S` | holds when every unread gap `g` has `b^g ≥ m` |
| `f_bot = 2` | `min(reach(b,m,L), min(m, b^(n-L)))`; both caps are 2 at `m=2` |
| `min Σ log₂ F = n−1` | plateau `m`, ramp `⌈log_b m⌉`; `n−1` is the `m=2` ramp-one case |
| four contact classes | two severings of a ring; on the ring one class, `f = 2L+1` |
| the carry interface is binary | the carry alphabet has size `m`, at every radix |

The method is the only part likely to transfer, so it is stated once. A law found in a
corner cannot be distinguished from a law about the whole family by inspection, by
review, or by agreement. It can only be distinguished by moving a parameter.

## 2. The machine family

Two orientations of the same loop. Loading the tape with digits and setting `c_0 = 0`
is addition. Loading `c_0` with a value and zeroing the tape is base conversion.

In [5]:
# the same loop, both loadings
print("b = 10, x = 2026, n = 4")
print("   converter: tape [0,0,0,0]  c0 = 2026  ->", loop([0,0,0,0], 2026, 10, 4)[0])
print("   adder:     tape [6,2,0,2]  c0 = 0     ->", loop([6,2,0,2], 0, 10, 4)[0])

bad = tot = 0
for b in (2, 3, 10, 16):
    for x in range(b ** 4):
        tot += 1
        bad += (loop([0]*4, x, b, 4)[0] != digits(x, b, 4))
print(f"\nconversion identity: {tot} values over b=2,3,10,16 at n=4, {bad} mismatches")

b = 10, x = 2026, n = 4
   converter: tape [0,0,0,0]  c0 = 2026  -> [6, 2, 0, 2]
   adder:     tape [6,2,0,2]  c0 = 0     -> [6, 2, 0, 2]

conversion identity: 75633 values over b=2,3,10,16 at n=4, 0 mismatches


## 3. The carry alphabet

**Proposition 1.** For `m` operands in base `b`, the carry takes exactly the values
`0 … m−1`, independently of `b` and of `n`.

If `c ≤ m−1` then `t + c ≤ m(b−1) + (m−1) = mb − 1`, so the next carry is at most
`⌊(mb−1)/b⌋ = m−1`; and every value in the range is achieved.

The previous work reported the carry interface as binary and treated that as a base-free
fact about addition. It is neither base-free nor about addition: it is the statement
that `m = 2`. A fixed operand count keeps the update bounded at any width and any radix;
bounded continuation fails only when `m` itself grows with the problem.

In [7]:
# the carry alphabet, by iterating the worst case to a fixed point
bases = (2, 3, 5, 10, 16, 256)
print("   m  " + "".join(f"{('b='+str(b)):>8}" for b in bases) + "     m-1")
bad = 0
for m in (1, 2, 3, 4, 7, 12):
    row = [max_carry(b, m) for b in bases]
    bad += sum(1 for c in row if c != m - 1)
    print(f"  {m:>3}  " + "".join(f"{c:>8}" for c in row) + f"{m-1:>8}")
print(f"\nmax carry = m-1 everywhere: {bad} deviations")

   m       b=2     b=3     b=5    b=10    b=16   b=256     m-1
    1         0       0       0       0       0       0       0
    2         1       1       1       1       1       1       1
    3         2       2       2       2       2       2       2
    4         3       3       3       3       3       3       3
    7         6       6       6       6       6       6       6
   12        11      11      11      11      11      11      11

max carry = m-1 everywhere: 0 deviations


## 4. The capacity theorem

### 4.1 The product law fails for m > 2

At `m = 2` the product law holds without exception. At `m = 3` it fails, and the
failures do not wash out with width — they grow. A finite-size artefact would thin.

In [9]:
# m = 2: the control
bad = tot = 0
for b, n in ((2,5), (2,6), (3,4), (4,4)):
    for S in cuts(n):
        tot += 1
        bad += (F_product(n, b, 2, S) != F(n, b, 2, S))
print(f"m=2 product law: {tot} cuts over (b,n)=(2,5),(2,6),(3,4),(4,4) -> {bad} mismatches")

# m = 3: the failures, and how they scale
print("\nm=3, b=2, failing cuts by width:")
for n in (4, 5, 6, 7):
    f = sum(1 for S in cuts(n) if F_product(n, 2, 3, S) != F(n, 2, 3, S))
    print(f"   n={n}: {f:>3} of {2**n:>4} cuts fail")
print("\nthey grow with n.  a finite-size artefact would thin.")

m=2 product law: 128 cuts over (b,n)=(2,5),(2,6),(3,4),(4,4) -> 0 mismatches

m=3, b=2, failing cuts by width:
   n=4:   2 of   16 cuts fail
   n=5:   7 of   32 cuts fail
   n=6:  18 of   64 cuts fail
   n=7:  42 of  128 cuts fail

they grow with n.  a finite-size artefact would thin.


   n=6:  18 of   64 cuts fail


   n=7:  42 of  128 cuts fail

they grow with n.  a finite-size artefact would thin.


### 4.2 The threshold, with the radix varied

Sorting failures by the smallest unread gap separates them completely. Writing `g*` for
the smallest gap above which no cut fails, and comparing against `⌈log_b m⌉`.

The rows that matter are the ones where a function of `m` alone and a function of
`b` and `m` disagree — which is why the radix had to be moved. Every earlier number
supporting this threshold was taken at `b = 2`, where `⌈log₂ m⌉` sends 2,3,4,5 to
1,2,2,3 and is indistinguishable from several functions of `m` alone.

In [11]:
# sort the failures by the smallest unread gap, and vary the radix
print(f"   {'b':>2} {'m':>2} {'n':>2} {'g* pred':>8} {'g* obs':>7} {'fails':>6}   "
      f"gap -> (holds, fails)")
match = 0; rows = 0
for (b, m, n) in ((2,2,6), (2,3,6), (2,4,6), (2,5,6),
                  (3,2,5), (3,3,5), (3,4,4), (3,5,4),
                  (4,3,4), (4,5,4), (5,5,3)):
    by = {}
    for S in cuts(n):
        ok = (F_product(n, b, m, S) == F(n, b, m, S))
        gs = gaps_of(S, n)
        g = min(gs) if gs else n
        by.setdefault(g, [0, 0])
        by[g][0 if ok else 1] += 1
    obs = min([g for g in sorted(by) if all(by[gg][1] == 0 for gg in by if gg >= g)])
    pred = gstar(b, m); rows += 1; match += (obs == pred)
    print(f"   {b:>2} {m:>2} {n:>2} {pred:>8} {obs:>7} {sum(v[1] for v in by.values()):>6}   "
          + "  ".join(f"g={g}:{tuple(by[g])}" for g in sorted(by))
          + ("   MATCH" if obs == pred else "   MISS"))
print(f"\n{match}/{rows} radix-operand pairs match ceil(log_b m)")
print("b=2,m=3 has 18 failures; b=3,m=3 has none.  same m, different radix.")

    b  m  n  g* pred  g* obs  fails   gap -> (holds, fails)
    2  2  6        1       1      0   g=1:(43, 0)  g=2:(10, 0)  g=3:(4, 0)  g=4:(3, 0)  g=5:(2, 0)  g=6:(2, 0)   MATCH
    2  3  6        2       2     18   g=1:(25, 18)  g=2:(10, 0)  g=3:(4, 0)  g=4:(3, 0)  g=5:(2, 0)  g=6:(2, 0)   MATCH
    2  4  6        2       2     27   g=1:(16, 27)  g=2:(10, 0)  g=3:(4, 0)  g=4:(3, 0)  g=5:(2, 0)  g=6:(2, 0)   MATCH
    2  5  6        3       3     29   g=1:(16, 27)  g=2:(8, 2)  g=3:(4, 0)  g=4:(3, 0)  g=5:(2, 0)  g=6:(2, 0)   MATCH
    3  2  5        1       1      0   g=1:(20, 0)  g=2:(5, 0)  g=3:(3, 0)  g=4:(2, 0)  g=5:(2, 0)   MATCH
    3  3  5        1       1      0   g=1:(20, 0)  g=2:(5, 0)  g=3:(3, 0)  g=4:(2, 0)  g=5:(2, 0)   MATCH
    3  4  4        2       2      2   g=1:(7, 2)  g=2:(3, 0)  g=3:(2, 0)  g=4:(2, 0)   MATCH
    3  5  4        2       2      4   g=1:(5, 4)  g=2:(3, 0)  g=3:(2, 0)  g=4:(2, 0)   MATCH
    4  3  4        1       1      0   g=1:(9, 0)  g=2:(3, 0)  g=

    2  3  6        2       2     18   g=1:(25, 18)  g=2:(10, 0)  g=3:(4, 0)  g=4:(3, 0)  g=5:(2, 0)  g=6:(2, 0)   MATCH


    2  4  6        2       2     27   g=1:(16, 27)  g=2:(10, 0)  g=3:(4, 0)  g=4:(3, 0)  g=5:(2, 0)  g=6:(2, 0)   MATCH


    2  5  6        3       3     29   g=1:(16, 27)  g=2:(8, 2)  g=3:(4, 0)  g=4:(3, 0)  g=5:(2, 0)  g=6:(2, 0)   MATCH


    3  2  5        1       1      0   g=1:(20, 0)  g=2:(5, 0)  g=3:(3, 0)  g=4:(2, 0)  g=5:(2, 0)   MATCH


    3  3  5        1       1      0   g=1:(20, 0)  g=2:(5, 0)  g=3:(3, 0)  g=4:(2, 0)  g=5:(2, 0)   MATCH


    3  4  4        2       2      2   g=1:(7, 2)  g=2:(3, 0)  g=3:(2, 0)  g=4:(2, 0)   MATCH


    3  5  4        2       2      4   g=1:(5, 4)  g=2:(3, 0)  g=3:(2, 0)  g=4:(2, 0)   MATCH


    4  3  4        1       1      0   g=1:(9, 0)  g=2:(3, 0)  g=3:(2, 0)  g=4:(2, 0)   MATCH


    4  5  4        2       2      2   g=1:(7, 2)  g=2:(3, 0)  g=3:(2, 0)  g=4:(2, 0)   MATCH


    5  5  3        1       1      0   g=1:(4, 0)  g=2:(2, 0)  g=3:(2, 0)   MATCH

11/11 radix-operand pairs match ceil(log_b m)
b=2,m=3 has 18 failures; b=3,m=3 has none.  same m, different radix.


### 4.3 The mechanism: a gap must be able to forget

The obvious reading of `b^g ≥ m` — a gap of `g` positions holds `b^g` words and must
cover `m` carry states — is wrong about this machine. A position holds a *digit sum*,
with `m(b−1)+1` values, not `b`. That count is at least `m` for every `g ≥ 1`, so it
would predict threshold 1 everywhere, which the table above refutes. **`b^g` is not
counting words.**

What decouples two runs is the gap's ability to make the downstream carry-in independent
of the upstream carry-out — to realise a *constant* transfer on the carry alphabet.

**Lemma (the forgetting gap).** A gap of `g` positions all carrying digit sum 0 maps
`c ↦ ⌊c / b^g⌋`. That map is constant on `{0,…,m−1}` if and only if `m−1 < b^g`.

So `b^g` is the divisor the gap applies to the carry. A run of zeros divides the
incoming carry by `b^g`; the gap severs two runs exactly when that division annihilates
the whole carry alphabet.

At `m = 2` the threshold is `⌈log_b 2⌉ = 1` for every `b ≥ 2`, and two distinct maximal
runs are separated by at least one unread position *by definition*. The hypothesis is
satisfied by every admissible cut. It reads as no hypothesis at all.

In [13]:
# which gaps can realise a CONSTANT carry transfer?
print(f"   {'b':>2} {'m':>2} {'g':>2} {'constants realisable':>22} {'b^g':>5} {'b^g>=m':>7}")
bad = 0
for b, m in ((2,2), (2,3), (2,4), (2,5), (3,3), (3,4), (3,5), (4,3), (4,5), (5,5), (10,3)):
    for g in (1, 2, 3):
        if (m * (b - 1) + 1) ** g > 200000: continue
        cs = constants(b, m, g)
        bad += (bool(cs) != (b ** g >= m))
        if (b, m) in ((2,3), (3,3), (2,5), (5,5)):
            print(f"   {b:>2} {m:>2} {g:>2} {str(cs):>22} {b**g:>5} {str(b**g >= m):>7}")
print(f"\nconstants exist  <=>  b^g >= m :  {bad} mismatches")

# and the one-line reason
bad = tot = 0
for b in (2, 3, 4, 5, 10):
    for m in range(2, 13):
        for g in (1, 2, 3, 4):
            tot += 1
            bad += ((len({c // b**g for c in range(m)}) == 1) != (b**g >= m))
print(f"floor(c / b^g) constant on {{0..m-1}}  <=>  b^g >= m :  {tot} triples, {bad} mismatches")

    b  m  g   constants realisable   b^g  b^g>=m
    2  3  1                     []     2   False
    2  3  2              [0, 1, 2]     4    True
    2  3  3              [0, 1, 2]     8    True
    2  5  1                     []     2   False
    2  5  2                     []     4   False
    2  5  3        [0, 1, 2, 3, 4]     8    True
    3  3  1              [0, 1, 2]     3    True
    3  3  2              [0, 1, 2]     9    True
    3  3  3              [0, 1, 2]    27    True
    5  5  1        [0, 1, 2, 3, 4]     5    True
    5  5  2        [0, 1, 2, 3, 4]    25    True
    5  5  3        [0, 1, 2, 3, 4]   125    True

constants exist  <=>  b^g >= m :  0 mismatches
floor(c / b^g) constant on {0..m-1}  <=>  b^g >= m :  220 triples, 0 mismatches


In [14]:
# 4.4  what the threshold is a property of
print("T1  the count is not the constraint")
print(f"   {'b':>2} {'m':>2} {'g':>2} {'words':>7} {'maps':>6} {'const':>6} {'b^g':>5}"
      f" {'maps>=m':>8}")
kill = 0
for b, m in ((2,3), (2,4), (2,5), (3,4), (3,5), (4,5), (2,8)):
    for g in (1, 2):
        if (m*(b-1)+1)**g > 100000: continue
        mm = transfer_maps(b, m, g)
        cs = [f for f in mm if len(set(f)) == 1]
        enough = len(mm) >= m
        kill += (enough and not cs)
        print(f"   {b:>2} {m:>2} {g:>2} {(m*(b-1)+1)**g:>7} {len(mm):>6} {len(cs):>6} "
              f"{b**g:>5} {str(enough):>8}"
              + ("   <- more maps than states, no constant" if enough and not cs else ""))
print(f"\n   {kill} cases offer at least m distinct transfers and cannot separate")

print("\nT2  is the all-zeros word optimal within the family?")
bad = tot = 0
for b in (2, 3, 4, 5):
    for m in range(2, 9):
        for g in (1, 2, 3):
            if (m*(b-1)+1)**g > 100000: continue
            any_const = any(len(set(f)) == 1 for f in transfer_maps(b, m, g))
            tot += 1
            bad += (any_const != (len(set(zeros_transfer(b, m, g))) == 1))
print(f"   {tot} (b,m,g) triples: a constant exists <=> the zeros word is constant, "
      f"{bad} mismatches")
print("   -> nothing beats the zeros word, so b^g >= m is NECESSARY and sufficient here")

print("\nT3  move the family, hold b, m, g")
print(f"   {'b':>2} {'m':>2} {'g':>2} {'b^g>=m':>7} {'adder family':>14} {'+reset':>10}")
for b, m in ((2,3), (2,5), (2,8), (3,5), (4,5)):
    cb = sorted({f[0] for f in transfer_maps(b, m, 1) if len(set(f)) == 1})
    ce = sorted({f[0] for f in transfer_maps(b, m, 1, True) if len(set(f)) == 1})
    print(f"   {b:>2} {m:>2} {1:>2} {str(b >= m):>7} {str(cb):>14} {str(ce):>10}")
print("\n   same b, m, g.  one extra admissible symbol and one position separates any m.")
print("   the threshold is a property of the realisable transfer FAMILY, not of width")
print("   or alphabet size.  the storage reading is refuted.")

T1  the count is not the constraint
    b  m  g   words   maps  const   b^g  maps>=m
    2  3  1       4      4      0     2     True   <- more maps than states, no constant
    2  3  2      16      7      3     4     True
    2  4  1       5      5      0     2     True   <- more maps than states, no constant
    2  4  2      25     13      4     4     True
    2  5  1       6      6      0     2     True   <- more maps than states, no constant
    2  5  2      36     16      0     4     True   <- more maps than states, no constant
    3  4  1       9      9      0     3     True   <- more maps than states, no constant
    3  4  2      81     13      4     9     True
    3  5  1      11     11      0     3     True   <- more maps than states, no constant
    3  5  2     121     21      5     9     True
    4  5  1      16     16      0     4     True   <- more maps than states, no constant
    4  5  2     256     21      5    16     True
    2  8  1       9      9      0     2     Tru

### 4.4 Status

The failure direction has an argument: when `b^g < m` no gap word realises a constant
transfer, so the two debts cannot be set independently and the product overcounts. The
lemma supplies the separation map on the other side.

What is **not** proved is that per-gap separation composes into the global product over
all runs simultaneously. Until that is closed the correct form is conditional:

> **Conjecture (capacity factorization).** For the `m`-operand base-`b` line machine,
> `F(S) = ∏ f(R)` whenever every gap of `S` has length at least `⌈log_b m⌉`. If some gap
> is shorter, the factorization *can* fail.

"can" is load-bearing: at `b=2, m=5, g=2` the split is 8 holding to 2 failing. A short
gap removes the guarantee; it does not force a failure.

## 5. What the corner hid

### 5.1 The bottom factor is two capacities, both at 2

`f_bot = 2` was read as the boundary being cheap because the origin pins everything
below it. Half right, and the constant is an accident of `m = 2`. Two quantities control
it: how many carry values a run of length `L` can **build** from `c_0 = 0`, and how many
the unread tail can **distinguish**.

    f_bot(L) = min( reach(b, m, L) , min(m, b^(n-L)) )

At `m = 2` both capacities sit at 2, neither is ever binding, and the factor reads as a
constant. This is the Section 4 capacity condition acting from both ends of a run
instead of across a gap.

In [17]:
# the two-sided bottom law
print(f"   {'b':>2} {'m':>2} {'n':>2} {'L':>2} {'f_bot':>6} {'reach':>6} {'tail cap':>9} {'pred':>5}")
bad = tot = 0
for b, m in ((2,2), (2,3), (2,4), (2,5), (3,2), (3,3), (3,4), (4,3), (4,5)):
    for n in (4, 5, 6):
        if (m * (b - 1) + 1) ** n > 200000: continue
        for L in range(1, n):
            f = F(n, b, m, list(range(L)))
            pred = f_bot_law(b, m, n, L)
            tot += 1; bad += (f != pred)
            if (b, m, n) in ((2,3,5), (2,5,5), (3,3,5)):
                print(f"   {b:>2} {m:>2} {n:>2} {L:>2} {f:>6} {reach(b,m,L):>6} "
                      f"{tail_cap(b,m,n-L):>9} {pred:>5}")
print(f"\nf_bot = min(reach, tail cap): {tot} bottom runs, {bad} mismatches")
print(f"at m=2 both caps sit at 2, so the factor reads as the constant 2")

    b  m  n  L  f_bot  reach  tail cap  pred
    2  3  5  1      2      2         3     2
    2  3  5  2      3      3         3     3
    2  3  5  3      3      3         3     3
    2  3  5  4      2      3         2     2
    2  5  5  1      3      3         5     3
    2  5  5  2      4      4         5     4
    2  5  5  3      4      5         4     4
    2  5  5  4      2      5         2     2
    3  3  5  1      3      3         3     3
    3  3  5  2      3      3         3     3
    3  3  5  3      3      3         3     3
    3  3  5  4      3      3         3     3

f_bot = min(reach, tail cap): 89 bottom runs, 0 mismatches
at m=2 both caps sit at 2, so the factor reads as the constant 2


    3  3  5  1      3      3         3     3
    3  3  5  2      3      3         3     3
    3  3  5  3      3      3         3     3
    3  3  5  4      3      3         3     3



f_bot = min(reach, tail cap): 89 bottom runs, 0 mismatches
at m=2 both caps sit at 2, so the factor reads as the constant 2


### 5.2 The n−1 profile is the ramp-one case

Reading least-significant-first and recording `F` at each step gives the cheapest
traversal's integrated cost. At `m = 2` the profile is `1, 2, 2, …, 2, 1` and the cost
is `n−1`. The plateau turns out to be `m` and the entrance ramp `⌈log_b m⌉` — the same
threshold as the gap condition, measured on a different object.

The `b = 3, m = 3` row is the discriminating one: the ramp collapses to 1 because one
base-3 position already covers a carry alphabet of 3. So `n−1` was never a law about
width.

In [19]:
# the LSB-first profile, by operand count
print(f"   {'b':>2} {'m':>2} {'n':>2}  {'profile':<22} {'plateau':>8} {'ramp':>5} "
      f"{'ceil(log_b m)':>14} {'sum log2':>9}")
import math as _m
okp = okr = tot = 0
for b, m, n in ((2,2,6), (2,3,6), (2,4,6), (2,5,6), (3,2,4), (3,3,4)):
    p = [F(n, b, m, list(range(t))) for t in range(n + 1)]
    ramp = next(i for i, v in enumerate(p) if v == max(p))
    tot += 1; okp += (max(p) == m); okr += (ramp == gstar(b, m))
    print(f"   {b:>2} {m:>2} {n:>2}  {str(p):<22} {max(p):>8} {ramp:>5} "
          f"{gstar(b,m):>14} {sum(_m.log2(x) for x in p):>9.4f}")
print(f"\nplateau = m in {okp}/{tot} rows;  ramp = ceil(log_b m) in {okr}/{tot} rows")

    b  m  n  profile                 plateau  ramp  ceil(log_b m)  sum log2
    2  2  6  [1, 2, 2, 2, 2, 2, 1]         2     1              1    5.0000
    2  3  6  [1, 2, 3, 3, 3, 2, 1]         3     2              2    6.7549
    2  4  6  [1, 3, 4, 4, 4, 2, 1]         4     2              2    8.5850
    2  5  6  [1, 3, 4, 5, 4, 2, 1]         5     3              3    8.9069
    3  2  4  [1, 2, 2, 2, 1]               2     1              1    3.0000
    3  3  4  [1, 3, 3, 3, 1]               3     1              1    4.7549

plateau = m in 6/6 rows;  ramp = ceil(log_b m) in 6/6 rows


    2  5  6  [1, 3, 4, 5, 4, 2, 1]         5     3              3    8.9069
    3  2  4  [1, 2, 2, 2, 1]               2     1              1    3.0000
    3  3  4  [1, 3, 3, 3, 1]               3     1              1    4.7549

plateau = m in 6/6 rows;  ramp = ceil(log_b m) in 6/6 rows


### 5.3 The contact classes are two severings

Run the identical reader on the **ring** machine — the same loop with the carry out of
the top re-entering at the bottom, which is addition mod `b^n − 1` — and both ends are
gone. If the four contact classes belong to the machine rather than the reader, they
must collapse.

They do. `F` becomes rotation-invariant, the per-run factor depends on length alone, and
the single surviving factor is `2L+1` — which on the line is the **interior** factor,
the class no boundary condition touches.

One further consequence: on the ring the full cut costs 2, not 1. The loop admits two
consistent carry assignments — the double zero of one's-complement. Complete reading is
free only on the severed line.

In [21]:
# the same reader on a machine with no ends
for n in (4, 5):
    Fr = {S: F(n, 2, 2, list(S), ring_out) for S in cuts(n)}
    Fl = {S: F(n, 2, 2, list(S), line_out) for S in cuts(n)}
    rot = lambda S: tuple(sorted((p + 1) % n for p in S))
    ri = sum(1 for S in cuts(n) if Fr[S] == Fr[rot(S)])
    li = sum(1 for S in cuts(n) if Fl[S] == Fl[rot(S)])
    print(f"n={n}: rotation-invariant   ring {ri}/{len(cuts(n))}   line {li}/{len(cuts(n))}")
    print(f"   {'L':>2} {'ring f':>7}   line f by contact")
    for L in range(1, n):
        bylen = {}
        for l in range(n):
            r = l + L - 1
            if r >= n: continue
            k = contact(l, r, n)
            if k == "full": continue
            bylen.setdefault(k, set()).add(Fl[tuple(range(l, r+1))])
        rv = {Fr[tuple(range(l, l+L))] for l in range(n - L + 1)}
        print(f"   {L:>2} {str(sorted(rv)):>7}   "
              + "  ".join(f"{k} {sorted(v)}" for k, v in sorted(bylen.items())))
    print(f"   full cut: ring {Fr[tuple(range(n))]}   line {Fl[tuple(range(n))]}\n")
print("contact is gone on the ring; the one surviving factor is 2L+1, the INTERIOR factor")

n=4: rotation-invariant   ring 16/16   line 6/16
    L  ring f   line f by contact
    1     [3]   bot [2]  int [3]  top [2]
    2     [5]   bot [2]  int [5]  top [3]
    3     [7]   bot [2]  top [4]
   full cut: ring 2   line 1

n=5: rotation-invariant   ring 32/32   line 10/32
    L  ring f   line f by contact
    1     [3]   bot [2]  int [3]  top [2]
    2     [5]   bot [2]  int [5]  top [3]
    3     [7]   bot [2]  int [7]  top [4]
    4     [9]   bot [2]  top [5]
   full cut: ring 2   line 1

contact is gone on the ring; the one surviving factor is 2L+1, the INTERIOR factor


## 6. The origin is a boundary condition, not a convention

Remove the lower severing alone: let `c_0` range freely, keep the top truncation.

The bottom factor does not become the top factor — it becomes the **interior** factor,
and top and interior do not move at all. The bottom run is cheap because the carry
beneath it is pinned, not because the carry runs upward. An unpinned bottom is an
interior; the top stays expensive because truncation pins nothing, it discards.

Two consequences for the previous work: its Proposition 1 depends on the origin, and so
does the equivalence between base-universal cuts and `F = 1` cuts.

In [23]:
# remove the lower severing only
n = 5
print(f"   {'run':<14} {'contact':<8} {'pinned c_0':>11} {'free c_0':>9}")
for (l, r) in ((0,0), (0,1), (0,2), (1,1), (1,2), (2,3), (3,4), (4,4), (0,n-1)):
    S = list(range(l, r+1))
    print(f"   [{l},{r}] L={r-l+1:<7} {contact(l,r,n):<8} "
          f"{F(n,2,2,S,line_out):>11} {F(n,2,2,S,free_out):>9}")

print("\nProposition 1, with and without the origin:")
for n in (4, 5, 6):
    pin = [S for S in cuts(n) if F(n,2,2,list(S),line_out) == 1]
    fre = [S for S in cuts(n) if F(n,2,2,list(S),free_out) == 1]
    print(f"   n={n}: F=1 pinned {len(pin)} cuts, free {len(fre)} cut;"
          f"  full cut free F = {F(n,2,2,list(range(n)),free_out)}  ((b-1)n+1 = {n+1})")

   run            contact   pinned c_0  free c_0
   [0,0] L=1       bot                2         3
   [0,1] L=2       bot                2         5
   [0,2] L=3       bot                2         7
   [1,1] L=1       int                3         3
   [1,2] L=2       int                5         5
   [2,3] L=2       int                5         5
   [3,4] L=2       top                3         3
   [4,4] L=1       top                2         2
   [0,4] L=5       full               1         6

Proposition 1, with and without the origin:
   n=4: F=1 pinned 2 cuts, free 1 cut;  full cut free F = 5  ((b-1)n+1 = 5)
   n=5: F=1 pinned 2 cuts, free 1 cut;  full cut free F = 6  ((b-1)n+1 = 6)
   n=6: F=1 pinned 2 cuts, free 1 cut;  full cut free F = 7  ((b-1)n+1 = 7)


   n=6: F=1 pinned 2 cuts, free 1 cut;  full cut free F = 7  ((b-1)n+1 = 7)


## 7. Where the value sits

Run the loop with digits on the tape and `c_0 = 0` and it is addition. Run the identical
loop with zeros on the tape and a value in `c_0` and it emits that value's base-`b`
digits. **Base conversion is the adder with the number in the carry register.**

| | tape holds | state holds | \|C\| | bounded? | value present? |
|---|---|---|---|---|---|
| adder | the value | a carry | `m`, every `n` and `b` | yes | only at the ends |
| converter | nothing | the value | `b^n` | no | always — it *is* the state |

A machine in which no value ever appears on the tape is a machine whose state is the
value, and that state grows with the word. `m` is not this switch; it sets the
granularity of the bounded side. The switch is the loading.

## 8. The resolution tower

Fix a value space of `2^N` and re-read it at base `2^k` for each `k | N`.

    c^(k)_q = c^(1)_{qk}

Grouping is associative, so the carry at a digit boundary cannot depend on how the
positions below it were bundled. A change of base does not move the motion; it changes
the stride at which the motion is read.

The sampling is not invertible and the loss is exactly uniform — a fibration with
constant fibre `2^(N−N/k)`. Coarsening composes and peeking is available at every rung,
but no rung is recoverable from the one above. The object is a **fibration, not a
stack**: a stack's pop undoes its push.

Which cuts a rung can name is a divisibility condition: `loc(S) = gcd(∂S ∪ {N})`, and
`S` survives regrouping at `k` exactly when `k | loc(S)`. Present in *every* reading:
exactly two cuts, at every `N` — and on the pinned line those are exactly the `F = 1`
cuts.

In [26]:
# the motion is one object, sampled
bad = tot = 0
for N in (8, 12):
    for x in range(2**N):
        for y in range(0, 2**N, max(1, 2**N // 32)):
            fine = carry_walk(x, y, 2, N)
            for k in divisors(N):
                if k == 1: continue
                tot += 1
                bad += (carry_walk(x, y, 2**k, N//k)
                        != tuple(fine[q*k] for q in range(N//k + 1)))
print(f"c^(k)_q = c^(1)_(qk):  {tot} trajectories, {bad} mismatches")
x = 0b101010101010
print("x = 0xAAA, N=12: " + "  ".join(f"k={k}:{carry_walk(x,x,2**k,12//k)}"
                                      for k in (1,2,3,4)))

# fibres
print()
for k in (2, 3, 4, 6):
    pre = {}
    for x in range(2**12):
        pre.setdefault(carry_walk(x, x, 2**k, 12//k), set()).add(carry_walk(x, x, 2, 12))
    print(f"   k={k}: {len(pre):>3} coarse walks, fibre {sorted({len(v) for v in pre.values()})}, "
          f"2^(N-N/k) = {2**(12 - 12//k)}")

# the tower, loc, and the universal cuts
print()
for N in (4, 6, 8, 12, 16):
    inter = set(cuts(N)); cov = set()
    for k in divisors(N):
        fam = {lift(T, k) for T in cuts(N // k)}
        inter &= fam
        if k > 1: cov |= fam
    print(f"   N={N:>2}: rungs {[len(cuts(N//k)) for k in divisors(N)]}  "
          f"universal {len(inter)}  finest-only {2**N - len(cov)} of {2**N}")

bad = tot = 0
for n in (4, 6, 8, 12):
    fams = {k: {lift(T, k) for T in cuts(n//k)} for k in divisors(n)}
    for S in cuts(n):
        g = loc(S, n)
        for k in fams:
            tot += 1
            bad += ((S in fams[k]) != (g % k == 0))
print(f"\nS survives regrouping at k  <=>  k | loc(S):  {tot} pairs, {bad} mismatches")

bad = tot = 0
for N in (4, 6):
    inter = set(cuts(N))
    for k in divisors(N): inter &= {lift(T, k) for T in cuts(N//k)}
    for S in cuts(N):
        tot += 1
        bad += ((S in inter) != (F(N, 2, 2, list(S)) == 1))
print(f"base-universal <=> F=1 (pinned line):  {tot} cuts, {bad} mismatches")

c^(k)_q = c^(1)_(qk):  679936 trajectories, 0 mismatches
x = 0xAAA, N=12: k=1:(0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1)  k=2:(0, 1, 1, 1, 1, 1, 1)  k=3:(0, 0, 1, 0, 1)  k=4:(0, 1, 1, 1)

   k=2:  64 coarse walks, fibre [64], 2^(N-N/k) = 64
   k=3:  16 coarse walks, fibre [256], 2^(N-N/k) = 256
   k=4:   8 coarse walks, fibre [512], 2^(N-N/k) = 512
   k=6:   4 coarse walks, fibre [1024], 2^(N-N/k) = 1024

   N= 4: rungs [16, 4, 2]  universal 2  finest-only 12 of 16
   N= 6: rungs [64, 8, 4, 2]  universal 2  finest-only 54 of 64
   N= 8: rungs [256, 16, 4, 2]  universal 2  finest-only 240 of 256
   N=12: rungs [4096, 64, 16, 8, 4, 2]  universal 2  finest-only 4020 of 4096
   N=16: rungs [65536, 256, 16, 4, 2]  universal 2  finest-only 65280 of 65536

S survives regrouping at k  <=>  k | loc(S):  25904 pairs, 0 mismatches
base-universal <=> F=1 (pinned line):  80 cuts, 0 mismatches


base-universal <=> F=1 (pinned line):  80 cuts, 0 mismatches


## 9. Negative results

Four claims tested and killed. A killed claim is a constraint.

**9.1 The free-carry machine is not an H2 witness.** Breaking H2 by removing the origin
leaves the machine one-way. Breaking H2 is necessary for a witness and is not sufficient:
an unresolved boundary condition *inside* the directed chain is not a coordinate
*outside* the index set.

**9.2 The previous work's machine witness is trivial.** The external-coordinate reader
has every `|Q_S| = 1`. It witnesses nothing. The only working witness is the abstract
coordinate-0 quotient — so the contrivance is load-bearing and should be presented so.

**9.3 Sampling is not a stabiliser.** A stabiliser is a subgroup; the sampling-invisible
set is not closed under the action. Same word for the blindness, different object.

**9.4 Flatness is not periodicity.** A flat sampled walk can come from repetition or
from nothing firing. The exact criterion is that every digit at that rung lies on the
same side of `B/2`.

In [28]:
# the four filtration signatures
print(f"   {'filtration':<30} {'H1':>5} {'H2':>6} {'nontriv':>8} {'backward-fn':>12}")
for n in (3, 4):
    for name, f in (("adder, pinned c_0", adder_filtration(n, False)),
                    ("adder, free c_0",   adder_filtration(n, True)),
                    ("coordinate-0 quotient", coord0_filtration(n)),
                    ("external-coord machine", external_filtration(n))):
        h1, h2, nt, bf = f.signature()
        print(f"   {name + ', n=' + str(n):<30} {str(h1):>5} {str(h2):>6} "
              f"{str(nt):>8} {str(bf):>12}")
print()
print("free c_0 breaks H2 and stays one-way: breaking H2 is necessary, not sufficient.")
print("the external-coordinate machine is TRIVIAL -- it witnesses nothing.")
print("only the abstract coordinate-0 quotient is a witness.")

   filtration                        H1     H2  nontriv  backward-fn
   adder, pinned c_0, n=3          True   True     True        False
   adder, free c_0, n=3            True  False     True        False
   coordinate-0 quotient, n=3      True  False     True         True
   external-coord machine, n=3     True   True    False         True
   adder, pinned c_0, n=4          True   True     True        False
   adder, free c_0, n=4            True  False     True        False
   coordinate-0 quotient, n=4      True  False     True         True
   external-coord machine, n=4     True   True    False         True

free c_0 breaks H2 and stays one-way: breaking H2 is necessary, not sufficient.
the external-coordinate machine is TRIVIAL -- it witnesses nothing.
only the abstract coordinate-0 quotient is a witness.


In [29]:
# 9.3  sampling blindness is not a subgroup
def flat(x, k, N):
    return len(set(carry_walk(x, x, 2**k, N//k)[1:])) <= 1

N = 12
flat2 = {x for x in range(2**N) if flat(x, 2, N)}
sub = sorted(flat2)[:128]
out = sum(1 for x in sub for y in sub if (x + y) % 2**N not in flat2)
print(f"9.3  rung 2: of {len(sub)**2} sums of two flat-walk values, "
      f"{out} leave the set -> NOT a subgroup")
for k in (1, 2, 5):
    M = 2**10
    D = {x: carry_walk(x, x, 2**k, 10//k) for x in range(M)}
    stab = [t for t in range(M) if all(D[(x+t) % M] == D[x] for x in range(M))]
    print(f"     rung k={k}: translation stabiliser of the sampled walk = {stab}")

# 9.4  flat is not the same as periodic
bad = tot = 0
for N in (8, 12, 16):
    for x in range(2**N):
        for k in divisors(N):
            B = 2**k
            ds = blocks(x, k, N)
            same = all(2*d < B for d in ds) or all(2*d >= B for d in ds)
            tot += 1; bad += (same != flat(x, k, N))
print(f"\n9.4  flat  <=>  every digit on the same side of B/2 :  "
      f"{tot} (value, rung) pairs, {bad} mismatches")

rep = other = 0
for x in range(2**12):
    p = period(x, 12)
    for k in divisors(12):
        if not flat(x, k, 12): continue
        if k % p == 0: rep += 1
        else: other += 1
print(f"     of the flat walks at N=12: {rep} flat because the value repeats, "
      f"{other} because nothing fires or everything does")

# open problem 5: paths grow in depth while the value does not
from collections import defaultdict as _dd
print("\nopen 5  distinct carry-walk sequences reaching ONE fixed value, by depth:")
for n, tgt in ((4, 0b1011), (5, 0b10110)):
    M = 1 << n; front = {0: {()}}; got = []
    for L in range(1, 7):
        nxt = _dd(set)
        for acc, seqs in front.items():
            for y in range(M):
                t = [((acc >> q) & 1) + ((y >> q) & 1) for q in range(n)]
                c = 0; w = [0]
                for q in range(n):
                    v = t[q] + c; c = v // 2; w.append(c)
                for s in seqs: nxt[(acc + y) % M].add(s + (tuple(w),))
        front = dict(nxt); got.append(len(front.get(tgt, set())))
    rate = got[-1] / got[-2]
    print(f"   n={n}: {got}   rate ~ {rate:.1f}   (the value is one point at every L)")

9.3  rung 2: of 16384 sums of two flat-walk values, 13464 leave the set -> NOT a subgroup
     rung k=1: translation stabiliser of the sampled walk = [0]
     rung k=2: translation stabiliser of the sampled walk = [0]
     rung k=5: translation stabiliser of the sampled walk = [0]

9.4  flat  <=>  every digit on the same side of B/2 :  353280 (value, rung) pairs, 0 mismatches
     of the flat walks at N=12: 4190 flat because the value repeats, 3620 because nothing fires or everything does

open 5  distinct carry-walk sequences reaching ONE fixed value, by depth:
   n=4: [1, 3, 30, 236, 1966, 15961]   rate ~ 8.1   (the value is one point at every L)
   n=5: [1, 11, 111, 1709, 23682, 336284]   rate ~ 14.2   (the value is one point at every L)


   n=4: [1, 3, 30, 236, 1966, 15961]   rate ~ 8.1   (the value is one point at every L)


   n=5: [1, 11, 111, 1709, 23682, 336284]   rate ~ 14.2   (the value is one point at every L)


## 10. Corrections

1. The carry interface is not base-independently binary. The alphabet has size `m` at every radix.
2. The product law is conditional on `b^g ≥ m` for every unread gap. At `m = 2` the hypothesis is invisible.
3. The bottom factor is not the constant 2 but `min(reach, tail cap)`.
4. `n−1` is not a law about width; it is plateau 2 with ramp 1.
5. The four contact classes are two severings of a ring.
6. Proposition 1 depends on the origin.
7. *(author)* The base-universal ⟺ `F = 1` equivalence was stated unconditionally in a draft of this work; it carries the origin hypothesis.
8. The external-coordinate H2 witness is trivial.
9. *(author)* A third operand does not destroy bounded update; it widens the bounded alphabet to 3.
10. *(author)* The threshold `⌈log_b m⌉` was stated on evidence taken entirely at `b = 2`.

Correction 10 is the one worth generalising. The threshold was correct. The evidence did
not distinguish it from several wrong alternatives, and three independent readings of
the draft reproduced the claim rather than testing it. **A pinned parameter is invisible
to agreement.** It becomes visible only when it is moved.

## 11. Open problems

1. **Composition.** The forgetting-gap lemma separates one gap. Show that per-gap separation composes into the global product, or exhibit a cut where every gap clears the threshold and the product still fails.
2. **The shape of the ramp.** Ramp length is `⌈log_b m⌉` and the plateau is `m`, but the intermediate values (1, 3, 4, 5 at `b=2, m=5`) are not derived.
3. **The ring's bistability.** The full cut costs 2 on the ring. Whether the ring admits a Proposition-1 analogue is open.
4. **Boundedness at growing `m`.** Everything here fixes `m`. The interesting regime for the fourth clause is `m = m(n)`.
5. **Retained paths.** Under composition the value is flat while distinct carry-walk sequences reaching a fixed target grow exponentially in depth. The path is not recoverable from the endpoint and this machine discards it.

## 12. A note on method

The same failure has now produced four results in this programme: a criterion clause
vacuous in every machine tested; a boundary condition read as a convention; base-2
magnitudes stated as base-free laws; and an operand count of 2 standing in for a family.

In each case the defective claim was not wrong in its domain and could not be found
wrong by checking it — checking it means evaluating it in that domain. It was found by
moving a parameter that had been held at a value nobody had noticed was a value.

The corollary is uncomfortable and is the reason for correction 10. Independent
agreement is not evidence about a pinned parameter, because agreement evaluates the
claim where the claim was made. **A second opinion tests the argument; only an
experiment tests the scope.**

## 13. Every check, re-run

The cell below re-runs every verification in this notebook and prints one line per
check. Any nonzero mismatch count is a failure of the paper, not of the machine.

In [34]:
# every check, re-run
checks = []

def chk(name, bad, tot):
    checks.append((name, bad, tot))

bad = tot = 0
for m in (1,2,3,4,7,12):
    for b in (2,3,5,10,16,256):
        tot += 1; bad += (max_carry(b, m) != m - 1)
chk("carry alphabet = m", bad, tot)

bad = tot = 0
for b, n in ((2,5),(2,6),(3,4),(4,4)):
    for S in cuts(n):
        tot += 1; bad += (F_product(n,b,2,S) != F(n,b,2,S))
chk("m=2 product law", bad, tot)

bad = tot = 0
for (b,m,n) in ((2,2,6),(2,3,6),(2,4,6),(2,5,6),(3,2,5),(3,3,5),
                (3,4,4),(3,5,4),(4,3,4),(4,5,4),(5,5,3)):
    by = {}
    for S in cuts(n):
        ok = (F_product(n,b,m,S) == F(n,b,m,S))
        gs = gaps_of(S,n); g = min(gs) if gs else n
        by.setdefault(g,[0,0]); by[g][0 if ok else 1] += 1
    obs = min([g for g in sorted(by) if all(by[gg][1]==0 for gg in by if gg>=g)])
    tot += 1; bad += (obs != gstar(b,m))
chk("capacity threshold = ceil(log_b m)", bad, tot)

bad = tot = 0
for b, m in ((2,2),(2,3),(2,4),(2,5),(3,3),(3,4),(3,5),(4,3),(4,5),(5,5),(10,3)):
    for g in (1,2,3):
        if (m*(b-1)+1)**g > 200000: continue
        tot += 1; bad += (bool(constants(b,m,g)) != (b**g >= m))
chk("constants exist <=> b^g >= m", bad, tot)

bad = tot = 0
for b in (2,3,4,5,10):
    for m in range(2,13):
        for g in (1,2,3,4):
            tot += 1
            bad += ((len({c//b**g for c in range(m)})==1) != (b**g >= m))
chk("floor(c/b^g) constant <=> b^g >= m", bad, tot)

bad = tot = 0
for b, m in ((2,2),(2,3),(2,4),(2,5),(3,2),(3,3),(3,4),(4,3),(4,5)):
    for n in (4,5,6):
        if (m*(b-1)+1)**n > 200000: continue
        for L in range(1, n):
            tot += 1
            bad += (F(n,b,m,list(range(L))) != f_bot_law(b,m,n,L))
chk("two-sided bottom law", bad, tot)

bad = tot = 0
for b, m, n in ((2,2,6),(2,3,6),(2,4,6),(2,5,6),(3,2,4),(3,3,4)):
    p = [F(n,b,m,list(range(t))) for t in range(n+1)]
    tot += 2
    bad += (max(p) != m)
    bad += (next(i for i,v in enumerate(p) if v == max(p)) != gstar(b,m))
chk("profile plateau = m, ramp = ceil(log_b m)", bad, tot)

bad = tot = 0
for n in (4,5):
    Fr = {S: F(n,2,2,list(S),ring_out) for S in cuts(n)}
    rot = lambda S: tuple(sorted((p+1)%n for p in S))
    for S in cuts(n):
        tot += 1; bad += (Fr[S] != Fr[rot(S)])
    for L in range(1, n):
        vals = {Fr[tuple(range(l,l+L))] for l in range(n-L+1)}
        tot += 1; bad += (vals != {2*L+1})
chk("ring: rotation-invariant, f = 2L+1", bad, tot)

bad = tot = 0
for n in (4,5,6):
    tot += 2
    bad += (len([S for S in cuts(n) if F(n,2,2,list(S),line_out)==1]) != 2)
    bad += (len([S for S in cuts(n) if F(n,2,2,list(S),free_out)==1]) != 1)
chk("Prop 1 pinned = 2 cuts, free = 1 cut", bad, tot)

bad = tot = 0
for n in (3,4):
    tot += 4
    bad += (adder_filtration(n, False).signature() != (True, True, True, False))
    bad += (adder_filtration(n, True ).signature() != (True, False, True, False))
    bad += (coord0_filtration(n).signature()       != (True, False, True, True))
    bad += (external_filtration(n).signature()     != (True, True, False, True))
chk("filtration signatures", bad, tot)

bad = tot = 0
for N in (8,12):
    for x in range(2**N):
        for y in range(0, 2**N, max(1, 2**N//32)):
            fine = carry_walk(x,y,2,N)
            for k in divisors(N):
                if k == 1: continue
                tot += 1
                bad += (carry_walk(x,y,2**k,N//k) != tuple(fine[q*k] for q in range(N//k+1)))
chk("c^(k)_q = c^(1)_(qk)", bad, tot)

bad = tot = 0
for n in (4,6,8,12):
    fams = {k: {lift(T,k) for T in cuts(n//k)} for k in divisors(n)}
    for S in cuts(n):
        g = loc(S,n)
        for k in fams:
            tot += 1; bad += ((S in fams[k]) != (g % k == 0))
chk("survives k <=> k | loc(S)", bad, tot)

bad = tot = 0
for N in (4,6):
    inter = set(cuts(N))
    for k in divisors(N): inter &= {lift(T,k) for T in cuts(N//k)}
    for S in cuts(N):
        tot += 1; bad += ((S in inter) != (F(N,2,2,list(S))==1))
chk("base-universal <=> F=1 (pinned)", bad, tot)

bad = tot = 0
for b in (2,3,10,16):
    for x in range(b**4):
        tot += 1; bad += (loop([0]*4, x, b, 4)[0] != digits(x, b, 4))
chk("conversion = adder with value in carry", bad, tot)

bad = tot = 0
for N in (8,12,16):
    for x in range((1<<N)-1):
        tot += 1; bad += (orbit_len(x,N) != period(x,N))
chk("orbit length = period", bad, tot)

print(f"{'check':<44} {'cases':>9} {'mismatches':>11}")
print("-" * 68)
for name, bad, tot in checks:
    print(f"{name:<44} {tot:>9} {bad:>11}")
print("-" * 68)
T = sum(t for _, _, t in checks); B = sum(b for _, b, _ in checks)
print(f"{'TOTAL':<44} {T:>9} {B:>11}")
print()
print("ALL CLEAR" if B == 0 else f"*** {B} MISMATCHES ***")

check                                            cases  mismatches
--------------------------------------------------------------------
carry alphabet = m                                  36           0
m=2 product law                                    128           0
capacity threshold = ceil(log_b m)                  11           0
constants exist <=> b^g >= m                        33           0
floor(c/b^g) constant <=> b^g >= m                 220           0
two-sided bottom law                                89           0
profile plateau = m, ramp = ceil(log_b m)           12           0
ring: rotation-invariant, f = 2L+1                  55           0
Prop 1 pinned = 2 cuts, free = 1 cut                 6           0
filtration signatures                                8           0
c^(k)_q = c^(1)_(qk)                            679936           0
survives k <=> k | loc(S)                        25904           0
base-universal <=> F=1 (pinned)                     80      